In [ ]:
from functools import partial

import matplotlib.pyplot as plt
import numpy as np

from expconfig.synthetic import (
    create_paths,
    gaussian_noise,
)
from sampling.priors import CompoundPrior
from tti.traveltimes import TravelTimeCalculator
from tti.traveltimes.parametrisations.nested_no_shear import (
    NestedNoShearDegreesParametriser,
)
from tti.traveltimes.paths import calculate_path_direction_vector

In [ ]:
from expconfig.synthetic import SynthConfig


def lonlatrad_to_xyz(lonlatrad: np.ndarray) -> np.ndarray:
    """Convert (lon, lat, radius) to Cartesian (x, y, z) coordinates.

    Parameters
    ----------
    lonlatrad : np.ndarray, shape (..., 3)
        Array of longitude (degrees), latitude (degrees), and radius (km).

    Returns
    -------
    xyz : np.ndarray, shape (..., 3)
        Array of Cartesian coordinates in km.
    """
    lon = np.radians(lonlatrad[..., 0])
    lat = np.radians(lonlatrad[..., 1])
    r = lonlatrad[..., 2]

    x = r * np.cos(lat) * np.cos(lon)
    y = r * np.cos(lat) * np.sin(lon)
    z = r * np.sin(lat)

    return np.stack([x, y, z], axis=-1)


def determine_weights(region, ic_in: np.ndarray, ic_out: np.ndarray) -> np.ndarray:
    """Determine weights for each path based on the distance travelled in each region.

    Parameters
    ----------
    region : CompositeRegion
        The composite region defining the geometry and properties of the inner core.
    ic_in : ndarray, shape (num_paths, 3)
        Entry points of paths into the inner core (longitude (deg), latitude (deg), radius (km)).
    ic_out : ndarray, shape (num_paths, 3)
        Exit points of paths from the inner core (longitude (deg), latitude (deg), radius

    Returns
    -------
    weights : ndarray, shape (1, num_segments, num_paths)
        Fractional distance of each path in each segment.  Additional axis for broadcasting with travel time calculator.
    """
    path_directions = calculate_path_direction_vector(ic_in, ic_out)
    segment_distances = region.ray_distances_per_region(
        lonlatrad_to_xyz(ic_in), path_directions
    )
    total_distances = segment_distances.sum(axis=1)
    weights = segment_distances / total_distances[:, None]
    return weights.T[None, ...]


cfg = SynthConfig.load("../config.yaml")
rng = np.random.default_rng(42)

prior = CompoundPrior.from_dict(cfg.priors.model_dump())
prior_samples = prior.sample(10_000, rng)

ic_in, ic_out = create_paths(source_spacing=30.0)
region = cfg.geometry.to_composite_region()
weights = determine_weights(region, ic_in, ic_out)
base_ttc_factory = partial(
    TravelTimeCalculator,
    ic_in=ic_in,
    ic_out=ic_out,
    normalisation=-0.5,
    weights=weights,
)

synth_calculator = base_ttc_factory()
synthetic_data_clean = synth_calculator(cfg.truth.as_array().flatten())[0]
noise = gaussian_noise(cfg.data.noise_level, rng, synthetic_data_clean)
synthetic_data = synthetic_data_clean + noise

ttc = base_ttc_factory(parametriser=NestedNoShearDegreesParametriser())
prior_predictive = ttc(prior_samples)
prior_predictive += gaussian_noise(cfg.data.noise_level, rng, prior_predictive)

In [ ]:
all_dists = np.vstack([prior_predictive, synthetic_data[None, :]])
hist_range = (all_dists.min(), all_dists.max())
common_kwargs = {
    "bins": 50,
    "histtype": "step",
    "range": hist_range,
    "density": True,
}
fig, ax = plt.subplots(
    1,
    1,
)
ax.hist(
    prior_predictive,
    **common_kwargs,
    color=np.full(prior_predictive.shape[1], "r"),
    alpha=0.3,
    label="Prior Sample Predictions",
)
ax.hist(
    synthetic_data,
    **common_kwargs,
    color="k",
    label="Synthetic Data",
)
ax.legend()

In [ ]:
zeta = np.arccos(ttc.path_directions[:, 2])
zeta[zeta > np.pi / 2] = np.pi - zeta[zeta > np.pi / 2]
zeta = np.degrees(zeta)

In [ ]:
N, M = prior_predictive.shape

# Repeat x for each prediction and flatten predicted values
x_all = np.tile(zeta, N)
p_all = prior_predictive.ravel()

# 2D histogram: x on horizontal axis, p on vertical axis
xbins = 50
pbins = 50

H, xedges, pedges = np.histogram2d(x_all, p_all, bins=[xbins, pbins], density=False)

# Optional: normalise each x-column so each x has comparable weight
H = H / np.maximum(H.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(9, 5))
img = plt.pcolormesh(xedges, pedges, H.T, shading="auto", cmap="Reds")
plt.errorbar(
    zeta,
    synthetic_data,
    yerr=cfg.data.noise_level,
    color="k",
    alpha=0.5,
    label="Data",
    fmt="none",
)
plt.xlabel("zeta (degrees)")
plt.ylabel("dt / t")
plt.title("Prior predictive density")
plt.colorbar(img, label="relative density")
plt.tight_layout()
plt.legend()